## 2.2 文档加载器 Document Loaders

数据源可能包含多种格式的文件，如文本文档、Markdown，PDF 等。LangChain 实现和集成了众多文档加载器（https://docs.langchain.com/oss/python/integrations/document_loaders ），方便从不同格式的文件中加载数据。

常用 Loaders：
* TextLoader - 文本文件
* CSVLoader - CSV 文件
* PyPDFLoader - PDF 文件
* WebBaseLoader - 网页

> ⚠️ **与最新版本不同：上面 4 个加载器都在 `langchain-community` 包里，这个包已经停止维护**
>
> * 2026-05-22，LangChain 在 GitHub 上宣布 `langchain-community` **停止维护**（sunset），立即生效（[langchain-community#674](https://github.com/langchain-ai/langchain-community/issues/674)）。公告说，这个包一年多来已经不接收新的集成和功能，维护也很有限；以后的集成改由各自独立的包提供。
> * 本项目装的 `langchain-community` 0.4.2 在第一次导入时会发出 `DeprecationWarning`，2.2.2 加载 CSV 那个单元格的输出里就有这条警告。代码照样能运行，只是以后不会再修复问题。
> * 官方 Document loaders 页面（上面的链接）**已经不再列出** `TextLoader`、`CSVLoader`、`JSONLoader`、`WebBaseLoader`，它们原来的说明页现在打开是 404（2025 年 12 月的网页存档里还能看到 JSONLoader 的说明页）。`PyPDFLoader` 还列在 PDFs 一类中，但标注为 Community。
> * 页面开头新加了一条警告：社区提供的加载器由用户贡献，官方没有审核，使用风险自负。“Common file types（常见文件类型）”一类现在推荐的是 Unstructured、Docling 这类能处理多种格式的加载器，它们有各自独立的包（`langchain-unstructured`、`langchain-docling`）。
>
> 对学习来说，继续用这几个加载器没有问题：所有加载器都实现同一个 `BaseLoader` 接口（见下一个单元格），换成别的加载器，`load()`、`lazy_load()` 的用法不变。

LangChain的设计：对于 Source 中多种不同的数据源，我们可以用一种统一的形式读取、调用。上述每一个文档加载器，都要继承自 `BaseLoader` 基类，此类提供了通用的 `load` （一次加载所有文档）与 `lazy_load` （以延迟方式加载文档） 方法，用于从数据源加载数据并处理为 Document 对象 。

#### 补充：`BaseLoader` 的源码

`BaseLoader` 定义在 `langchain_core/document_loaders/base.py`，`langchain_community.document_loaders.base` 只是把它重新导出。源码里的分工是：**子类只负责实现 `lazy_load()`，其他方法都由 `BaseLoader` 基于 `lazy_load()` 实现。** 所以上面说的“`BaseLoader` 提供 `lazy_load` 方法”，准确地说是 `BaseLoader` 规定了这个方法，真正读取数据的代码写在每个子类里。

![BaseLoader 的方法](assets/04-baseloader-methods.svg)

| 方法 | 参数 | 返回值 | 源码做了什么 |
|---|---|---|---|
| `lazy_load()` | 无 | `Iterator[Document]`（生成器） | 由子类实现，每 `yield` 一次产出一个 `Document`。`BaseLoader` 里的默认版本只会报 `NotImplementedError` |
| `load()` | 无 | `list[Document]` | `return list(self.lazy_load())`：把生成器一次取完。源码注释说子类不要重写它 |
| `alazy_load()` | 无 | `AsyncIterator[Document]` | 用 `run_in_executor` 在线程池里逐个调用 `lazy_load` 的 `next()`，供 `async for` 使用 |
| `aload()` | 无 | `list[Document]` | `load()` 的异步版：收集 `alazy_load()` 的全部结果 |
| `load_and_split(text_splitter=None)` | `text_splitter`：文本切分器，不传时用 `RecursiveCharacterTextSplitter()` | `list[Document]`（切分后的块） | 先 `load()`，再调用 `text_splitter.split_documents()`。源码注释写着：应视为已弃用 |

需要注意：

* 加载器的参数（文件路径、编码等）都在**创建对象**时传入，`load()`、`lazy_load()` 本身不接收参数。
* “延迟加载”的意思是：生成器每被 `next()` 一次，才产出一个 `Document`。`load()` 返回时，所有 `Document` 都已经在内存里；`for doc in loader.lazy_load()` 可以处理完一个再取下一个。只有一个小文件时两者没有区别；文件很大（例如几十万行的 CSV，每行一个 `Document`）或文件很多时，`lazy_load()` 才能省内存。2.2.1 后面有验证代码。
* `load_and_split` 已不推荐使用，切分交给文本切分器单独完成（RAG 流程的环节3）。

### 2.2.1 加载txt

In [2]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader(
    file_path="load/01-langchain-utf-8.txt",
    encoding="utf-8",
)

docs = loader.load()

print(docs)

[Document(metadata={'source': 'load/01-langchain-utf-8.txt'}, page_content='LangChain 是一个用于构建基于大语言模型（LLM）应用的开发框架，旨在帮助开发者更高效地集成、管理和增强大语言模型的能力，构建端到端的应用程序。它提供了一套模块化工具和接口，支持从简单的文本生成到复杂的多步骤推理任务')]


这里主要有2个类需要关注:
1. TextLoader

```python
TextLoader(
  self,
  file_path: Union[str, Path],
  encoding: Optional[str] = None,
  autodetect_encoding: bool = False
)
```

签名和当前源码（`langchain-community` 0.4.2）一致。

| 参数 | 类型 / 默认值 | 含义 |
|---|---|---|
| `file_path` | `str` 或 `Path`，必填 | 要读取的文件路径。会原样写进 `metadata["source"]`：传相对路径，`source` 就是相对路径 |
| `encoding` | `str`，默认 `None` | 打开文件时用的编码。`None` 表示用 Python 的默认编码：macOS、Linux 上一般是 UTF-8，中文 Windows 上一般是 GBK。同一份代码换一台电脑，结果可能不同，所以最好显式写出来 |
| `autodetect_encoding` | `bool`，默认 `False` | 按 `encoding` 解码失败时，是否自动检测文件编码，再按检测结果（可信度从高到低）逐个重试。检测要用 `chardet` 包，本项目的 `.venv` 里**没有安装**，打开它会报 `ModuleNotFoundError` |

**返回值**：`TextLoader` 的 `lazy_load()` 只 `yield` **一个** `Document`，所以 `load()` 返回的列表长度总是 1：

* `page_content`：整个文件的内容（`f.read()` 的结果）；
* `metadata`：`{"source": str(file_path)}`。

**出错时**：解码失败（且没有打开 `autodetect_encoding`）、文件不存在等错误，都会被包装成 `RuntimeError("Error loading <路径>")`，真正的原因在异常的 `__cause__` 里。

2. Document

```python
Document(
    self,
    page_content: str,
    kwargs: Any = {},
)
```

> ⚠️ **与当前源码不同**：签名里的 `kwargs: Any = {}` 应为 `**kwargs: Any`（任意个关键字参数）。`Document` 是 Pydantic 模型，`**kwargs` 用来传它从父类 `BaseMedia` 继承的两个字段：`id` 和 `metadata`。笔记的签名里看不到这两个字段，但它们才是 `kwargs` 里实际要传的内容。早期版本的 `Document` 没有 `id` 字段，较早的资料里也看不到它。

当前源码（`langchain-core` 1.6.6 的 `langchain_core/documents/base.py`，只保留字段和构造方法）：

```python
class BaseMedia(Serializable):
    id: str | None = Field(default=None, coerce_numbers_to_str=True)
    metadata: dict[Any, Any] = Field(default_factory=dict)

class Document(BaseMedia):
    page_content: str
    type: Literal["Document"] = "Document"

    def __init__(self, page_content: str, **kwargs: Any) -> None:
        super().__init__(page_content=page_content, **kwargs)
```

| 参数 / 字段 | 类型 / 默认值 | 含义 |
|---|---|---|
| `page_content` | `str`，必填 | 文本内容，可以按位置传：`Document("...")`。传入的不是字符串会报 `ValidationError` |
| `metadata`（通过 `**kwargs` 传） | `dict`，默认 `{}` | 任意元数据，例如来源、页码、行号。加载器会自动填入 `source` 等字段 |
| `id`（通过 `**kwargs` 传） | `str` 或 `None`，默认 `None` | 文档的标识，可选。传数字会自动转成字符串（`coerce_numbers_to_str=True`）。源码注释说以后可能改成必填 |
| `type` | 固定为 `"Document"` | 序列化时标明类型，不需要自己传 |

**返回值**：得到一个 `Document` 对象，各加载器返回的就是它的列表。下面 `print(type(docs[0]))` 的输出 `langchain_core.documents.base.Document` 说明：`Document` 定义在 `langchain-core` 里，不受 `langchain-community` 停止维护的影响。

In [3]:
print(type(docs[0]))

<class 'langchain_core.documents.base.Document'>


#### 验证：`encoding` 和 `autodetect_encoding`

`load/01-langchain-gbk.txt` 和上面读的是同一段文字，只是用 GBK 编码保存。下面分别用 UTF-8、GBK 和自动检测去读它：

In [ ]:
from langchain_community.document_loaders import TextLoader

gbk_path = "load/01-langchain-gbk.txt"

# 1. 用 UTF-8 读 GBK 编码的文件：解码失败，被包装成 RuntimeError
try:
    TextLoader(file_path=gbk_path, encoding="utf-8").load()
except RuntimeError as e:
    print("报错：", e)
    print("原因：", type(e.__cause__).__name__, "-", e.__cause__)

# 2. 指定正确的编码
docs = TextLoader(file_path=gbk_path, encoding="gbk").load()
print("encoding='gbk'：", docs[0].page_content[:20])

# 3. 自动检测编码：依赖 chardet 包，本环境没有安装
try:
    TextLoader(file_path=gbk_path, encoding="utf-8", autodetect_encoding=True).load()
except ModuleNotFoundError as e:
    print("autodetect_encoding=True：", e)

报错： Error loading load/01-langchain-gbk.txt
原因： UnicodeDecodeError - 'utf-8' codec can't decode byte 0xca in position 10: invalid continuation byte
encoding='gbk'： LangChain 是一个用于构建基于大
autodetect_encoding=True： No module named 'chardet'


* 用 UTF-8 解码，在 `position 10` 处失败：前 10 个字节 `LangChain ` 是 ASCII 字符，在两种编码里相同；从第一个中文字开始，两种编码就不同了。
* 报出的是 `RuntimeError`，原始的 `UnicodeDecodeError` 在 `__cause__` 里。
* `autodetect_encoding=True` 只有在 UTF-8 解码失败后才会去检测编码，这时才发现缺少 `chardet`。

#### 验证：`lazy_load()` 和 `load()` 的关系

调用 `lazy_load()` 只得到一个生成器，函数体还没有执行；第一次 `next()` 时才打开文件、读取内容、`yield` 出 `Document`。`TextLoader` 只产出一个 `Document`，所以第二次 `next()` 就没有了：

In [ ]:
loader = TextLoader(file_path="load/01-langchain-utf-8.txt", encoding="utf-8")

gen = loader.lazy_load()
print(type(gen))          # 生成器：此时函数体还没执行，文件还没打开

doc = next(gen)           # 第一次 next()：打开文件、读取、yield 一个 Document
print(doc.metadata)

print(next(gen, "没有更多 Document 了"))   # TextLoader 只产出 1 个

# load() 就是 list(lazy_load())
print(loader.load() == list(loader.lazy_load()))

<class 'generator'>
{'source': 'load/01-langchain-utf-8.txt'}
没有更多 Document 了
True


#### 验证：`Document` 的参数

`id` 传数字会被转成字符串。`**kwargs` 里写错的字段名**不会报错**，会被直接忽略（`Document` 的 Pydantic 配置是 `extra='ignore'`），所以把 `metadata` 拼错时很难发现：

In [ ]:
from langchain_core.documents import Document

doc = Document(
    "退货政策：七天无理由退货",     # page_content 可以按位置传
    metadata={"source": "faq.md"},
    id=1001,                        # 数字会被转成字符串
)
print(repr(doc))

typo = Document("同样的内容", meta={"source": "faq.md"})   # 把 metadata 写错成 meta
print(repr(typo))                                           # 不报错，meta 被忽略

Document(id='1001', metadata={'source': 'faq.md'}, page_content='退货政策：七天无理由退货')
Document(metadata={}, page_content='同样的内容')


### 2.2.2 加载CSV

In [2]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(
    file_path="load/02-load.csv",
)

docs = loader.load()
print(docs)

[Document(metadata={'source': 'load/02-load.csv', 'row': 0}, page_content='id: 1\ntitle: Introduction to Python\ncontent: Python is a popular programming language.\nauthor: John Doe'), Document(metadata={'source': 'load/02-load.csv', 'row': 1}, page_content='id: 2\ntitle: Data Science Basics\ncontent: Data science involves statistics and machine learning.\nauthor: Jane Smith'), Document(metadata={'source': 'load/02-load.csv', 'row': 2}, page_content='id: 3\ntitle: Web Development\ncontent: HTML, CSS and JavaScript are core web technologies.\nauthor: Mike Johnson'), Document(metadata={'source': 'load/02-load.csv', 'row': 3}, page_content='id: 4\ntitle: Artificial Intelligence\ncontent: AI is transforming many industries.\nauthor: Sarah Williams')]


/var/folders/14/yh05p4nj2z70mjs568rzs9gh0000gn/T/ipykernel_80771/2455859707.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import CSVLoader


In [3]:
from rich import print as rprint
for doc in docs:
    rprint(doc)

Document(
    metadata={'source': 'load/02-load.csv', 'row': 0},
    page_content='id: 1\ntitle: Introduction to Python\ncontent: Python is a popular programming language.\nauthor:
John Doe'
)

Document(
    metadata={'source': 'load/02-load.csv', 'row': 1},
    page_content='id: 2\ntitle: Data Science Basics\ncontent: Data science involves statistics and machine 
learning.\nauthor: Jane Smith'
)

Document(
    metadata={'source': 'load/02-load.csv', 'row': 2},
    page_content='id: 3\ntitle: Web Development\ncontent: HTML, CSS and JavaScript are core web 
technologies.\nauthor: Mike Johnson'
)

Document(
    metadata={'source': 'load/02-load.csv', 'row': 3},
    page_content='id: 4\ntitle: Artificial Intelligence\ncontent: AI is transforming many industries.\nauthor: 
Sarah Williams'
)

#### CSVLoader 的参数与返回值

笔记没有列出 `CSVLoader` 的源码，这里补上（`langchain-community` 0.4.2）：

```python
CSVLoader(
    self,
    file_path: Union[str, Path],
    source_column: Optional[str] = None,
    metadata_columns: Sequence[str] = (),
    csv_args: Optional[Dict] = None,
    encoding: Optional[str] = None,
    autodetect_encoding: bool = False,
    *,
    content_columns: Sequence[str] = (),
)
```

| 参数 | 类型 / 默认值 | 含义 |
|---|---|---|
| `file_path` | `str` 或 `Path`，必填 | CSV 文件路径 |
| `source_column` | `str`，默认 `None` | 用哪一列的值作为每行的 `metadata["source"]`。默认所有行都用文件路径。列名不存在会报错 |
| `metadata_columns` | 列名序列，默认 `()` | 这些列放进 `metadata`，并且不再出现在 `page_content` 里 |
| `content_columns` | 列名序列，默认 `()` | 只有这些列拼进 `page_content`。默认为空，表示用 `metadata_columns` 以外的所有列。它在签名的 `*` 之后，只能用关键字传 |
| `csv_args` | `dict`，默认 `None` | 原样传给 Python 标准库的 `csv.DictReader`，例如 `delimiter`（分隔符）、`quotechar`（引号字符）、`fieldnames`（列名）。传了 `fieldnames` 时，文件第一行不再当作表头，会被当成数据 |
| `encoding`、`autodetect_encoding` | 同 `TextLoader` | 同 `TextLoader` |

**返回值**：每一行数据生成一个 `Document`（表头不算）：

* `page_content`：每列一行，格式是 `列名: 值`，值两端的空白会被去掉。上面输出里的 `'id: 1\ntitle: ...'` 就是这样拼出来的；
* `metadata`：`{"source": 文件路径或 source_column 列的值, "row": 行号}`，行号从 0 开始；再加上 `metadata_columns` 里的列。

上面第一个 CSV 单元格输出里的 `DeprecationWarning`，就是 2.2 开头说的 `langchain-community` 停止维护的提示，每个内核会话里第一次导入时出现一次。

下面用 `source_column`、`metadata_columns`、`content_columns` 三个参数，调整 `page_content` 和 `metadata` 的内容：

In [ ]:
loader = CSVLoader(
    file_path="load/02-load.csv",
    source_column="title",              # source 改用 title 列的值
    metadata_columns=["id", "author"],  # 这两列进 metadata，不再出现在 page_content 里
    content_columns=["content"],        # 只有 content 列拼进 page_content
)

doc = loader.load()[0]
print(repr(doc.page_content))
print(doc.metadata)

'content: Python is a popular programming language.'
{'source': 'Introduction to Python', 'row': 0, 'id': '1', 'author': 'John Doe'}


### 2.2.3 加载JSON
LangChain提供的JSON格式的文档加载器是 JSONLoader 。在实际应用场景中，JSON格式的数据占有很大比例，而且JSON的形式也是多样的。我们需要特别关注。

JSONLoader 使用指定的 jq结构 来解析 JSON 文件。jq是一个轻量级的命令行 JSON 处理器 ，可以对JSON 格式的数据进行各种复杂的处理，包括数据过滤、映射、减少和转换，是处理 JSON 数据的 首选工具之一 。

补充：`JSONLoader` 用的是 Python 的 `jq` 包（`pip install jq`，本项目装的是 1.12.0），不是 jq 命令行工具。没装这个包时，创建 `JSONLoader` 对象就会报 `ImportError`。

笔记两种情况用到的 jq 写法（以 `load/03-load.json` 为例）：

| jq 表达式 | 含义 | 产出的结果 |
|---|---|---|
| `.` | 整个 JSON | 1 个：整个对象（dict） |
| `.messages` | 取 `messages` 字段 | 1 个：整个列表（list） |
| `.messages[]` | 取 `messages`，再把列表展开 | 4 个：每条消息（dict） |
| `.messages[].content` | 展开后，取每条消息的 `content` | 4 个：每条消息的文本（str） |

**jq 产出几个结果，`JSONLoader` 就生成几个 `Document`。** 所以 `.messages` 和 `.messages[]` 只差两个字符，前者得到 1 个 `Document`，后者得到 4 个。

> ⚠️ **与最新文档不同**：官方原来有一个 JSONLoader 的说明页（2025 年 12 月的网页存档里还在），现在已经删除，打开是 404；`JSONLoader` 也在停止维护的 `langchain-community` 里（见 2.2 开头）。下面的签名和当前源码一致。

```python
JSONLoader(
  self,
  file_path: Union[str, PathLike],
  jq_schema: str,
  content_key: Optional[str] = None,
  is_content_key_jq_parsable: Optional[bool] = False,
  metadata_func: Optional[Callable[[Dict, Dict], Dict]] = None,
  text_content: bool = True,
  json_lines: bool = False
)
```

| 参数 | 类型 / 默认值 | 含义 |
|---|---|---|
| `file_path` | `str` 或 `PathLike`，必填 | JSON 文件路径。源码先用 `Path(file_path).resolve()` 把它转成**绝对路径**，所以 `metadata["source"]` 是绝对路径（`TextLoader` 是原样保留）。没有 `encoding` 参数，固定按 UTF-8 读取（文件开头可以带 BOM） |
| `jq_schema` | `str`，必填 | jq 表达式，决定从 JSON 里挑出哪些数据。产出几个结果，就生成几个 `Document` |
| `content_key` | `str`，默认 `None` | jq 的结果是 dict 时，取哪个键的值作为 `page_content`。不设置时，整个结果就是内容。设置了它，jq 的结果就必须是 dict（源码只检查第一个结果），否则报 `ValueError` |
| `is_content_key_jq_parsable` | `bool`，默认 `False` | 为 `True` 时，`content_key` 被当成 jq 表达式，例如 `".author.name"`，可以取嵌套字段；为 `False` 时只当作普通的键名 |
| `metadata_func` | 函数，默认 `None` | 形如 `(record: dict, metadata: dict) -> dict`。`record` 是 jq 的一个结果，`metadata` 是默认元数据。函数的返回值会**整个替换**默认元数据，所以通常是在传入的 `metadata` 上添加字段，再把它返回；返回的不是 dict 会报错 |
| `text_content` | `bool`，默认 `True` | 内容是否必须是字符串。`True`：不是 `str` 就报 `ValueError`；`False`：dict、list 用 `json.dumps` 转成 JSON 字符串，数字等用 `str()` 转换 |
| `json_lines` | `bool`，默认 `False` | 文件是不是 JSON Lines 格式（每行一个 JSON）。为 `True` 时逐行解析、跳过空行，`seq_num` 跨行连续编号 |

**返回值**：`lazy_load()` 为 jq 的每个结果 `yield` 一个 `Document`：

* `page_content`：经过 `content_key`、`text_content` 处理后的字符串；
* `metadata`：默认是 `{"source": 文件的绝对路径, "seq_num": 序号}`，`seq_num` 从 1 开始（`CSVLoader` 的 `row` 从 0 开始）；设置了 `metadata_func` 时，是它的返回值。

几个参数分别在哪一步起作用：

![JSONLoader 的处理流程](assets/05-jsonloader-flow.svg)

In [9]:
# 1.导入依赖
from  langchain_community.document_loaders import JSONLoader
from rich import print as rprint

#2.定义JSONLoader对象
#情况1
json_loader=JSONLoader(
    file_path="load/03-load.json",
    jq_schema=".", #直接提取完整的JSON对象（包括所有字段）
    text_content=False #保持原始 JSON 结构，将提取的数据转换为JSON字符串存入page_content字段中
)

# 3.加载
docs = json_loader.load()
rprint(docs)
rprint(docs[0].page_content) #输出page_content字段内容

[
    Document(
        metadata={
            'source': '/Users/skk/Developer/lang-chain-learn/Chapter10_RAG/load/03-load.json',
            'seq_num': 1
        },
        page_content='{"messages": [{"sender": "Alice", "content": "Hello, how are you today?", "timestamp": 
"2023-05-15T10:00:00"}, {"sender": "Bob", "content": "I\'m doing well, thanks for asking!", "timestamp": 
"2023-05-15T10:02:00"}, {"sender": "Alice", "content": "Would you like to meet for lunch?", "timestamp": 
"2023-05-15T10:05:00"}, {"sender": "Bob", "content": "Sure, that sounds great!", "timestamp": 
"2023-05-15T10:07:00"}], "conversation_id": "conv_12345", "participants": ["Alice", "Bob"]}'
    )
]

{"messages": [{"sender": "Alice", "content": "Hello, how are you today?", "timestamp": "2023-05-15T10:00:00"}, 
{"sender": "Bob", "content": "I'm doing well, thanks for asking!", "timestamp": "2023-05-15T10:02:00"}, {"sender": 
"Alice", "content": "Would you like to meet for lunch?", "timestamp": "2023-05-15T10:05:00"}, {"sender": "Bob", 
"content": "Sure, that sounds great!", "timestamp": "2023-05-15T10:07:00"}], "conversation_id": "conv_12345", 
"participants": ["Alice", "Bob"]}

#### 情况1 为什么要写 `text_content=False`

`jq_schema="."` 的结果是整个对象（dict），而 `text_content` 默认为 `True`，要求内容是字符串，所以不写 `text_content=False` 会报错。写了以后，dict 用 `json.dumps` 转成字符串，这就是代码注释里说的“将提取的数据转换为 JSON 字符串存入 page_content”。

`json.dumps` 默认 `ensure_ascii=True`，**中文会被转义成 `\uXXXX`**。`03-load.json` 都是英文，看不出来；换成中文就很明显。这样的 `page_content` 交给嵌入模型，模型看到的是一串转义码而不是中文，语义就丢了。`JSONLoader` 没有参数能改变这一点，所以中文数据最好像情况2那样，用 jq 直接取出字符串字段。

In [ ]:
import json
import os
import tempfile

# 1. 情况1 不写 text_content=False：jq 的结果是 dict，不是 str，报错
try:
    JSONLoader(file_path="load/03-load.json", jq_schema=".").load()
except ValueError as e:
    print("ValueError：", " ".join(str(e).split()))

# 2. text_content=False 时，dict 用 json.dumps 转成字符串：中文变成了 \u 转义
with tempfile.NamedTemporaryFile("w", suffix=".json", encoding="utf-8", delete=False) as f:
    json.dump({"title": "退货政策", "body": "七天无理由退货"}, f, ensure_ascii=False)

doc = JSONLoader(file_path=f.name, jq_schema=".", text_content=False).load()[0]
os.remove(f.name)
print(doc.page_content)

ValueError： Expected page_content is string, got <class 'dict'> instead. Set `text_content=False` if the desired input for `page_content` is not a string
{"title": "\u9000\u8d27\u653f\u7b56", "body": "\u4e03\u5929\u65e0\u7406\u7531\u9000\u8d27"}


In [6]:
# 1.导入依赖
from  langchain_community.document_loaders import JSONLoader
from rich import print as rprint


# 情况2
# .messages[].content:遍历.messages[]中所有元素 从每一个元素中提取.content字段
json_loader=JSONLoader(
    file_path="load/03-load.json",
    jq_schema=".messages[].content"
)

# 3.加载
docs = json_loader.load()
rprint(docs)

[
    Document(
        metadata={
            'source': '/Users/skk/Developer/lang-chain-learn/Chapter10_RAG/load/03-load.json',
            'seq_num': 1
        },
        page_content='Hello, how are you today?'
    ),
    Document(
        metadata={
            'source': '/Users/skk/Developer/lang-chain-learn/Chapter10_RAG/load/03-load.json',
            'seq_num': 2
        },
        page_content="I'm doing well, thanks for asking!"
    ),
    Document(
        metadata={
            'source': '/Users/skk/Developer/lang-chain-learn/Chapter10_RAG/load/03-load.json',
            'seq_num': 3
        },
        page_content='Would you like to meet for lunch?'
    ),
    Document(
        metadata={
            'source': '/Users/skk/Developer/lang-chain-learn/Chapter10_RAG/load/03-load.json',
            'seq_num': 4
        },
        page_content='Sure, that sounds great!'
    )
]

#### 情况2 的结果

* jq 产出 4 个字符串，所以得到 4 个 `Document`，`seq_num` 依次是 1–4；
* 结果都是 `str`，默认的 `text_content=True` 能通过检查；
* `source` 是绝对路径，因为 `JSONLoader` 先对 `file_path` 做了 `resolve()`。

情况2 只保留了消息内容，谁发的、什么时候发的都丢掉了。用情况1、2 没有用到的 `content_key` 和 `metadata_func`，可以把一条消息拆开存放：`content` 进 `page_content`，`sender`、`timestamp` 进 `metadata`：

In [ ]:
def add_sender(record: dict, metadata: dict) -> dict:
    # record：jq 产出的一个结果，这里是一条消息（dict）
    # metadata：默认元数据 {"source": ..., "seq_num": ...}
    metadata["sender"] = record["sender"]
    metadata["timestamp"] = record["timestamp"]
    return metadata

json_loader = JSONLoader(
    file_path="load/03-load.json",
    jq_schema=".messages[]",      # 每条消息（dict）是一个结果
    content_key="content",        # 取每条消息的 content 作为 page_content
    metadata_func=add_sender,     # 把 sender、timestamp 加进 metadata
)

for doc in json_loader.load():
    m = doc.metadata
    print(m["seq_num"], m["sender"], m["timestamp"], "|", doc.page_content)

1 Alice 2023-05-15T10:00:00 | Hello, how are you today?
2 Bob 2023-05-15T10:02:00 | I'm doing well, thanks for asking!
3 Alice 2023-05-15T10:05:00 | Would you like to meet for lunch?
4 Bob 2023-05-15T10:07:00 | Sure, that sounds great!


In [10]:
# 1.导入相关依赖
from langchain_community.document_loaders import JSONLoader
from rich import print as rprint

# 2.定义json文件的路径
file_path = 'load/03-response.json'

#3.定义JSONLoader对象
#需求1：提取data.items中的数据
loader = JSONLoader(
    file_path=file_path,  # 文件路径
    jq_schema=".data.items[]",
    text_content=False,  # 提取内容是否为字符串格式
)


# 需求2：提取data.items[].content中的数据
# loader = JSONLoader(
#     file_path=file_path,  # 文件路径
#     jq_schema=".data.items[].content",
# )


# 需求3：提取data.items中指定字段的数据
# loader = JSONLoader(
#     file_path=file_path,  # 文件路径
#     jq_schema="""
#         .data.items[] | {
#             author,
#             created_at,
#             content: (.title + "\n" + .content)
#         }
#     """,
#     text_content=False,  # 提取内容是否为字符串格式
# )


# 4.加载
data = loader.load()
rprint(data)


[
    Document(
        metadata={
            'source': '/Users/skk/Developer/lang-chain-learn/Chapter10_RAG/load/03-response.json',
            'seq_num': 1
        },
        page_content='{"id": 101, "title": "Understanding JSONLoader", "content": "This article explains how to 
parse API responses...", "author": {"id": "user_1", "name": "Alice"}, "created_at": "2023-10-05T08:12:33Z"}'
    ),
    Document(
        metadata={
            'source': '/Users/skk/Developer/lang-chain-learn/Chapter10_RAG/load/03-response.json',
            'seq_num': 2
        },
        page_content='{"id": 102, "title": "Advanced jq Schema Patterns", "content": "Learn to handle nested 
structures with...", "author": {"id": "user_2", "name": "Bob"}, "created_at": "2023-10-05T09:15:21Z"}'
    ),
    Document(
        metadata={
            'source': '/Users/skk/Developer/lang-chain-learn/Chapter10_RAG/load/03-response.json',
            'seq_num': 3
        },
        page_content='{"id": 103, "title": "LangChain Metadata Handling", "content": "Best practices for preserving
metadata...", "author": {"id": "user_3", "name": "Charlie"}, "created_at": "2023-10-05T10:03:47Z"}'
    )
]

In [11]:
# 1.导入相关依赖
from langchain_community.document_loaders import JSONLoader
from rich import print as rprint

# 2.定义json文件的路径
file_path = 'load/03-response.json'

#3.定义JSONLoader对象
# #需求1：提取data.items中的数据
# loader = JSONLoader(
#     file_path=file_path,  # 文件路径
#     jq_schema=".data.items[]",
#     text_content=False,  # 提取内容是否为字符串格式
# )


# 需求2：提取data.items[].content中的数据
loader = JSONLoader(
    file_path=file_path,  # 文件路径
    jq_schema=".data.items[].content",
)


# 需求3：提取data.items中指定字段的数据
# loader = JSONLoader(
#     file_path=file_path,  # 文件路径
#     jq_schema="""
#         .data.items[] | {
#             author,
#             created_at,
#             content: (.title + "\n" + .content)
#         }
#     """,
#     text_content=False,  # 提取内容是否为字符串格式
# )


# 4.加载
data = loader.load()
rprint(data)


[
    Document(
        metadata={
            'source': '/Users/skk/Developer/lang-chain-learn/Chapter10_RAG/load/03-response.json',
            'seq_num': 1
        },
        page_content='This article explains how to parse API responses...'
    ),
    Document(
        metadata={
            'source': '/Users/skk/Developer/lang-chain-learn/Chapter10_RAG/load/03-response.json',
            'seq_num': 2
        },
        page_content='Learn to handle nested structures with...'
    ),
    Document(
        metadata={
            'source': '/Users/skk/Developer/lang-chain-learn/Chapter10_RAG/load/03-response.json',
            'seq_num': 3
        },
        page_content='Best practices for preserving metadata...'
    )
]

In [12]:
# 1.导入相关依赖
from langchain_community.document_loaders import JSONLoader
from rich import print as rprint

# 2.定义json文件的路径
file_path = 'load/03-response.json'

#3.定义JSONLoader对象
# #需求1：提取data.items中的数据
# loader = JSONLoader(
#     file_path=file_path,  # 文件路径
#     jq_schema=".data.items[]",
#     text_content=False,  # 提取内容是否为字符串格式
# )


# 需求2：提取data.items[].content中的数据
# loader = JSONLoader(
#     file_path=file_path,  # 文件路径
#     jq_schema=".data.items[].content",
# )


# 需求3：提取data.items中指定字段的数据
loader = JSONLoader(
    file_path=file_path,  # 文件路径
    jq_schema="""
        .data.items[] | {
            author,
            created_at,
            content: (.title + "\n" + .content)
        }
    """,
    text_content=False,  # 提取内容是否为字符串格式
)


# 4.加载
data = loader.load()
rprint(data)


[
    Document(
        metadata={
            'source': '/Users/skk/Developer/lang-chain-learn/Chapter10_RAG/load/03-response.json',
            'seq_num': 1
        },
        page_content='{"author": {"id": "user_1", "name": "Alice"}, "created_at": "2023-10-05T08:12:33Z", 
"content": "Understanding JSONLoader\\nThis article explains how to parse API responses..."}'
    ),
    Document(
        metadata={
            'source': '/Users/skk/Developer/lang-chain-learn/Chapter10_RAG/load/03-response.json',
            'seq_num': 2
        },
        page_content='{"author": {"id": "user_2", "name": "Bob"}, "created_at": "2023-10-05T09:15:21Z", "content": 
"Advanced jq Schema Patterns\\nLearn to handle nested structures with..."}'
    ),
    Document(
        metadata={
            'source': '/Users/skk/Developer/lang-chain-learn/Chapter10_RAG/load/03-response.json',
            'seq_num': 3
        },
        page_content='{"author": {"id": "user_3", "name": "Charlie"}, "created_at": "2023-10-05T10:03:47Z", 
"content": "LangChain Metadata Handling\\nBest practices for preserving metadata..."}'
    )
]